# SFT on a Base Model: Math Reasoning

**Goal:** Same base model (`Qwen3-4B`), different task — math word problems.  
**What changes:** The dataset format. Instead of `user/assistant` chat turns, math datasets use a structured template with `### Instruction`, `### Input`, `### Response` fields (Alpaca format). The model learns to produce a step-by-step solution under `### Response`.  
**Key insight:** SFT behaviour is shaped entirely by the data format — the same model can be steered toward chat, math, code, or any other task by changing what examples you show it.

## 1. Install dependencies

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## 2. Load the base model

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen3-4B",  # same base model as before — no instruct tuning
    max_seq_length = 2048,
    load_in_4bit   = True,
    load_in_8bit   = False,
    full_finetuning = False,
)

## 3. Baseline inference — before training

Give the base model a math word problem using the Alpaca prompt structure.  
It will attempt to complete the text but won't reliably produce a correct, structured solution.

In [ ]:
# Alpaca prompt template — this is what every training example will look like
# EOS_TOKEN is appended during training so the model learns when to stop generating
EOS_TOKEN = tokenizer.eos_token

ALPACA_TEMPLATE = """### Instruction:
{instruction}

### Input:
{input}

### Response:
{response}"""

# At inference we leave Response blank — the model must fill it in
INFERENCE_TEMPLATE = """### Instruction:
{instruction}

### Input:
{input}

### Response:
"""

test_problem = "Solve the math word problem step by step."
test_input   = "Janet has 3 bags of apples. Each bag contains 8 apples. She gives 5 apples to her friend. How many apples does Janet have left?"

raw_prompt = INFERENCE_TEMPLATE.format(instruction=test_problem, input=test_input)

inputs = tokenizer(raw_prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    out_ids = model.generate(
        **inputs,
        max_new_tokens = 150,
        temperature    = 0.7,
        top_p          = 0.9,
        do_sample      = True,
    )

before_response = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print("=== BASE MODEL (before SFT) ===")
print(before_response)

## 4. Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    lora_alpha     = 16,
    lora_dropout   = 0.0,
    bias           = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
    use_rslora     = False,
)

## 5. Load & prepare the math dataset

We use **GSM8K** (Grade School Math 8K) — 8,500 elementary math word problems, each with a step-by-step solution.  

### What the data looks like — raw vs formatted

Notice the format difference vs the previous notebook: there is no `user`/`assistant` here.  
Instead we use three plain-text fields that the model learns to associate as a unit.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main", split="train")

# Raw example — as it arrives from HuggingFace
print("--- RAW ---")
print(dataset[0])

In [ ]:
def format_examples(examples):
    """Render each GSM8K row into the Alpaca template string used for training."""
    texts = []
    for question, answer in zip(examples["question"], examples["answer"]):
        text = ALPACA_TEMPLATE.format(
            instruction = "Solve the math word problem step by step.",
            input       = question,
            response    = answer,
        ) + EOS_TOKEN   # EOS tells the model where generation should stop
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(format_examples, batched=True)

# After formatting — this is exactly what the model is trained on
print("--- AFTER ALPACA TEMPLATE ---")
print(dataset[0]["text"])

## 6. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field          = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # effective batch = 8
        warmup_steps                = 5,
        max_steps                   = 60,  # increase to num_train_epochs=1 for a full run
        learning_rate               = 2e-4,
        lr_scheduler_type           = "linear",
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        logging_steps               = 1,
        seed                        = 42,
        output_dir                  = "./sft_math_output",
        report_to                   = "none",
    ),
)

# Note: we do NOT use train_on_responses_only here.
# The Alpaca format is a single flat string — there are no role tokens to mask on.
# The model learns from the entire sequence including the ### headers.
trainer_stats = trainer.train()

In [ ]:
gpu_props = torch.cuda.get_device_properties(0)
total_mem = round(gpu_props.total_memory / 1024**3, 2)
peak_mem  = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print(f"Training time : {round(trainer_stats.metrics['train_runtime'] / 60, 2)} min")
print(f"Peak VRAM     : {peak_mem} GB  ({round(peak_mem/total_mem*100, 1)}% of {total_mem} GB)")

## 7. Post-training inference & comparison

In [ ]:
from transformers import TextStreamer

# Same problem as baseline — only the Response field is empty
prompt = INFERENCE_TEMPLATE.format(instruction=test_problem, input=test_input)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print("=== FINE-TUNED MODEL (after SFT on GSM8K) ===")
out_ids = model.generate(
    **inputs,
    max_new_tokens = 300,
    temperature    = 0.7,
    top_p          = 0.9,
    do_sample      = True,
    streamer       = TextStreamer(tokenizer, skip_prompt=True),
)

after_response = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

In [ ]:
print("PROBLEM:", test_input)
print()
print("━" * 60)
print("BEFORE SFT — base model, unstructured continuation:")
print("━" * 60)
print(before_response)
print()
print("━" * 60)
print("AFTER SFT — trained on GSM8K, step-by-step solution:")
print("━" * 60)
print(after_response)

## 8. Try your own problem

In [ ]:
def solve(problem, max_new_tokens=300):
    """Run a math word problem through the fine-tuned model."""
    prompt = INFERENCE_TEMPLATE.format(
        instruction = "Solve the math word problem step by step.",
        input       = problem,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7, top_p=0.9, do_sample=True,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )

solve("A train travels at 60 km/h. How long does it take to cover 210 km?")

## 9. Save LoRA adapters

In [ ]:
model.save_pretrained("qwen3_math_lora")
tokenizer.save_pretrained("qwen3_math_lora")

# model.push_to_hub("your_username/qwen3_math_lora", token="YOUR_HF_TOKEN")
# model.save_pretrained_merged("qwen3_math_merged", tokenizer, save_method="merged_16bit")

# Multi GPU Train

In [ ]:
%%writefile /kaggle/working/train_math.py

import os
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

def log(msg):
    if int(os.environ.get("LOCAL_RANK", 0)) == 0:
        print(msg)

log("=" * 60)
log("LOADING BASE MODEL")
log("=" * 60)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "unsloth/Qwen3-4B",
    max_seq_length  = 2048,
    load_in_4bit    = True,
    load_in_8bit    = False,
    full_finetuning = False,
)
log("Base model loaded!\n")

log("=" * 60)
log("ATTACHING LoRA ADAPTERS")
log("=" * 60)

model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    lora_alpha     = 16,
    lora_dropout   = 0.0,
    bias           = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
    use_rslora     = False,
)
log("LoRA adapters attached!\n")

EOS_TOKEN = tokenizer.eos_token

ALPACA_TEMPLATE = """### Instruction:
{instruction}

### Input:
{input}

### Response:
{response}"""

log("=" * 60)
log("LOADING & FORMATTING DATASET")
log("=" * 60)

def format_examples(examples):
    texts = []
    for question, answer in zip(examples["question"], examples["answer"]):
        text = ALPACA_TEMPLATE.format(
            instruction = "Solve the math word problem step by step.",
            input       = question,
            response    = answer,
        ) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = load_dataset("openai/gsm8k", "main", split="train")
log(f"Dataset loaded — {len(dataset)} examples")
dataset = dataset.map(format_examples, batched=True)
log("Formatting done!\n")

log("=" * 60)
log("SETTING UP TRAINER")
log("=" * 60)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field          = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps                = 5,
        max_steps                   = 60,
        learning_rate               = 2e-4,
        lr_scheduler_type           = "linear",
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        logging_steps               = 1,
        seed                        = 42,
        output_dir                  = "./sft_math_output",
        report_to                   = "none",
        ddp_find_unused_parameters  = False,
    ),
)
log("Trainer ready!\n")

log("=" * 60)
log("STARTING TRAINING")
log("=" * 60)

trainer_stats = trainer.train()

gpu_props = torch.cuda.get_device_properties(0)
total_mem = round(gpu_props.total_memory / 1024**3, 2)
peak_mem  = round(torch.cuda.max_memory_reserved() / 1024**3, 2)

log("\n" + "=" * 60)
log("TRAINING COMPLETE")
log("=" * 60)
log(f"Training time : {round(trainer_stats.metrics['train_runtime'] / 60, 2)} min")
log(f"Peak VRAM     : {peak_mem} GB  ({round(peak_mem / total_mem * 100, 1)}% of {total_mem} GB)\n")

log("=" * 60)
log("SAVING LoRA ADAPTERS")
log("=" * 60)

model.save_pretrained("qwen3_math_lora")
tokenizer.save_pretrained("qwen3_math_lora")
log("Adapters saved to qwen3_math_lora/\n")

In [ ]:
!torchrun --nproc_per_node=2 /kaggle/working/train_math.py